# Main Pipeline — Seasonal Profile + Residual LightGBM

## Mục tiêu refactor
- Giữ nguyên ý tưởng modeling chính của notebook gốc.
- Bỏ phần import / data load / export thừa hoặc lặp lại.
- Gom toàn bộ feature engineering vào helper functions để train và forecast dùng chung một logic.
- Làm rõ từng bước: seasonal baseline, residual model, validation, retrain, forecast.

## Pipeline
1. Dùng **2020-2022** để xây seasonal profile gần hiện tại.
2. Ước lượng yearly level bằng **YoY factor** từ 2022 sang 2023-2024.
3. Train **LightGBM trên residual** để học phần sai số quanh seasonal baseline.
4. Grid-search trên năm 2022 để chọn `YoY` và trọng số blend `w_lgbm`.
5. Retrain trên toàn bộ 2014-2022 rồi forecast 2023-2024.


In [1]:
from pathlib import Path
import warnings

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error

warnings.filterwarnings('ignore')

try:
    from google.colab import drive
except ImportError:
    drive = None

USE_COLAB_DRIVE = drive is not None
COLAB_DATA_DIR = Path('/content/drive/MyDrive/Colab Notebooks/DATATHON 2026/DATA/datathon-2026-round-1')
LOCAL_DATA_DIR = Path('dataset')
COLAB_OUT_FILE = Path('/content/drive/MyDrive/Colab Notebooks/DATATHON 2026/DATA/SOURCE/TRAIN/6/submission_main_pipeline_refactored.csv')
LOCAL_OUT_FILE = Path('submission_main_pipeline_refactored.csv')

if USE_COLAB_DRIVE:
    drive.mount('/content/drive')

DATA_DIR = COLAB_DATA_DIR if COLAB_DATA_DIR.exists() else LOCAL_DATA_DIR
OUT_FILE = COLAB_OUT_FILE if COLAB_OUT_FILE.parent.exists() else LOCAL_OUT_FILE
FC_START = '2023-01-01'
FC_END = '2024-07-01'
STABLE_YEARS = [2020, 2021, 2022]
TRAIN_START = '2014-01-01'
TRAIN_END = '2021-12-31'
VAL_START = '2022-01-01'
VAL_END = '2022-12-31'
FULL_TRAIN_END = '2022-12-31'
LGBM_WEIGHT_GRID = [0.0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0]

TET_DATES = {
    2013: '2013-02-10', 2014: '2014-01-31', 2015: '2015-02-19', 2016: '2016-02-08',
    2017: '2017-01-28', 2018: '2018-02-16', 2019: '2019-02-05', 2020: '2020-01-25',
    2021: '2021-02-12', 2022: '2022-02-01', 2023: '2023-01-22', 2024: '2024-02-10',
}

BASE_FEATURES = [
    'dow', 'month', 'day', 'quarter', 'week',
    'is_weekend', 'days_to_eom', 'is_eom', 'is_bom',
    'is_vn_holiday', 'is_tet', 'is_pre_tet', 'is_post_tet',
    'year_parity',
    'is_promo', 'promo_discount', 'is_urban', 'is_yearend', 'is_midyear', 'is_spring',
]
REV_LAG_FEATURES = ['resid_rev_lag364', 'resid_rev_lag365', 'resid_rev_roll7', 'resid_rev_roll30']
COGS_LAG_FEATURES = ['resid_cogs_lag364', 'resid_cogs_lag365', 'resid_cogs_roll7', 'resid_cogs_roll30']

print(f'DATA_DIR: {DATA_DIR}')
print(f'OUT_FILE: {OUT_FILE}')


DATA_DIR: dataset
OUT_FILE: submission_main_pipeline_refactored.csv


In [2]:
def load_sales(data_dir: Path) -> pd.DataFrame:
    sales = pd.read_csv(data_dir / 'sales.csv', parse_dates=['Date'])
    sales = sales.sort_values('Date').reset_index(drop=True)
    sales['year'] = sales['Date'].dt.year
    sales['month'] = sales['Date'].dt.month
    sales['day'] = sales['Date'].dt.day
    return sales


def build_seasonal_profile(sales: pd.DataFrame, stable_years: list[int]):
    stable = sales[sales['year'].isin(stable_years)].copy()
    stable['year_mean_rev'] = stable.groupby('year')['Revenue'].transform('mean')
    stable['year_mean_cogs'] = stable.groupby('year')['COGS'].transform('mean')
    stable['rev_norm'] = stable['Revenue'] / stable['year_mean_rev']
    stable['cogs_norm'] = stable['COGS'] / stable['year_mean_cogs']

    seasonal = (
        stable.groupby(['month', 'day'], as_index=False)[['rev_norm', 'cogs_norm']]
        .mean()
    )
    monthly_profile = (
        stable.groupby('month', as_index=False)[['rev_norm', 'cogs_norm']]
        .mean()
    )
    return stable, seasonal, monthly_profile


def get_yoy_candidates(yearly_mean_rev: pd.Series) -> dict[str, float]:
    yoy_cagr_2yr = (yearly_mean_rev[2022] / yearly_mean_rev[2020]) ** 0.5
    yoy_last_1yr = yearly_mean_rev[2022] / yearly_mean_rev[2021]
    yoy_flat = 1.0
    yoy_sample = 0.971
    return {
        'cagr_2yr': yoy_cagr_2yr,
        'last_1yr': yoy_last_1yr,
        'flat': yoy_flat,
        'sample_implied': yoy_sample,
        'avg_3': np.mean([yoy_cagr_2yr, yoy_last_1yr, yoy_flat]),
    }


def build_promo_calendar(start_year: int = 2012, end_year: int = 2024) -> pd.DataFrame:
    rows = []
    for year in range(start_year, end_year + 1):
        promos = [
            ('spring', f'{year}-03-18', f'{year}-04-17', 12),
            ('midyear', f'{year}-06-23', f'{year}-07-22', 18),
            ('fall', f'{year}-08-30', f'{year}-10-01', 10),
            ('yearend', f'{year}-11-18', f'{year + 1}-01-02', 20),
        ]
        if year % 2 == 1:
            promos.extend([
                ('urban', f'{year}-07-30', f'{year}-09-02', 50),
                ('rural', f'{year}-01-30', f'{year}-03-01', 15),
            ])
        for promo_type, start, end, discount in promos:
            rows.append((year, promo_type, pd.to_datetime(start), pd.to_datetime(end), discount))
    return pd.DataFrame(rows, columns=['year', 'promo_type', 'start', 'end', 'discount'])


def mark_promo_days(dates: pd.Series, promo_calendar: pd.DataFrame) -> pd.DataFrame:
    promo_flags = pd.DataFrame({
        'Date': pd.to_datetime(dates),
        'is_promo': 0,
        'promo_discount': 0.0,
        'is_urban': 0,
        'is_yearend': 0,
        'is_midyear': 0,
        'is_spring': 0,
    })
    for _, row in promo_calendar.iterrows():
        mask = (promo_flags['Date'] >= row['start']) & (promo_flags['Date'] <= row['end'])
        promo_flags.loc[mask, 'is_promo'] = 1
        promo_flags.loc[mask, 'promo_discount'] = np.maximum(
            promo_flags.loc[mask, 'promo_discount'], row['discount']
        )
        if row['promo_type'] in ['urban', 'yearend', 'midyear', 'spring']:
            promo_flags.loc[mask, f'is_{row["promo_type"]}'] = 1
    return promo_flags


def add_calendar_features(frame: pd.DataFrame) -> pd.DataFrame:
    enriched = frame.copy()
    dt = pd.to_datetime(enriched['Date'])
    enriched['dow'] = dt.dt.dayofweek
    enriched['quarter'] = dt.dt.quarter
    enriched['week'] = dt.dt.isocalendar().week.astype(int)
    enriched['is_weekend'] = (dt.dt.dayofweek >= 5).astype(int)

    days_to_eom = dt.dt.days_in_month - dt.dt.day
    enriched['days_to_eom'] = days_to_eom
    enriched['is_eom'] = (days_to_eom <= 4).astype(int)
    enriched['is_bom'] = (dt.dt.day <= 3).astype(int)
    enriched['year_parity'] = dt.dt.year % 2

    enriched['is_vn_holiday'] = (
        ((dt.dt.month == 1) & (dt.dt.day == 1)) |
        ((dt.dt.month == 4) & (dt.dt.day == 30)) |
        ((dt.dt.month == 5) & (dt.dt.day == 1)) |
        ((dt.dt.month == 9) & (dt.dt.day == 2))
    ).astype(int)

    enriched['is_tet'] = 0
    enriched['is_pre_tet'] = 0
    enriched['is_post_tet'] = 0
    for _, tet_start in TET_DATES.items():
        tet_day = pd.to_datetime(tet_start)
        tet_end = tet_day + pd.Timedelta(days=4)
        enriched.loc[(dt >= tet_day) & (dt <= tet_end), 'is_tet'] = 1
        enriched.loc[(dt >= tet_day - pd.Timedelta(days=7)) & (dt < tet_day), 'is_pre_tet'] = 1
        enriched.loc[(dt > tet_end) & (dt <= tet_end + pd.Timedelta(days=5)), 'is_post_tet'] = 1
    return enriched


def add_promo_features(frame: pd.DataFrame, promo_calendar: pd.DataFrame) -> pd.DataFrame:
    promo_flags = mark_promo_days(frame['Date'], promo_calendar)
    return frame.merge(promo_flags, on='Date', how='left')


def add_residual_lag_features(frame: pd.DataFrame) -> pd.DataFrame:
    enriched = frame.copy().sort_values('Date').reset_index(drop=True)
    enriched['resid_rev_lag364'] = enriched['residual_rev'].shift(364)
    enriched['resid_rev_lag365'] = enriched['residual_rev'].shift(365)
    enriched['resid_rev_roll7'] = enriched['residual_rev'].shift(364).rolling(7).mean()
    enriched['resid_rev_roll30'] = enriched['residual_rev'].shift(364).rolling(30).mean()
    enriched['resid_cogs_lag364'] = enriched['residual_cogs'].shift(364)
    enriched['resid_cogs_lag365'] = enriched['residual_cogs'].shift(365)
    enriched['resid_cogs_roll7'] = enriched['residual_cogs'].shift(364).rolling(7).mean()
    enriched['resid_cogs_roll30'] = enriched['residual_cogs'].shift(364).rolling(30).mean()
    return enriched


def build_master_table(sales: pd.DataFrame, seasonal: pd.DataFrame, promo_calendar: pd.DataFrame):
    yearly_mean_rev = sales.groupby('year')['Revenue'].mean().to_dict()
    yearly_mean_cogs = sales.groupby('year')['COGS'].mean().to_dict()

    master = sales.merge(seasonal, on=['month', 'day'], how='left')
    master['rev_norm'] = master['rev_norm'].fillna(1.0)
    master['cogs_norm'] = master['cogs_norm'].fillna(1.0)
    master['year_mean_rev'] = master['year'].map(yearly_mean_rev)
    master['year_mean_cogs'] = master['year'].map(yearly_mean_cogs)
    master['seasonal_pred_rev'] = master['rev_norm'] * master['year_mean_rev']
    master['seasonal_pred_cogs'] = master['cogs_norm'] * master['year_mean_cogs']
    master['residual_rev'] = master['Revenue'] - master['seasonal_pred_rev']
    master['residual_cogs'] = master['COGS'] - master['seasonal_pred_cogs']

    master = add_calendar_features(master)
    master = add_promo_features(master, promo_calendar)
    master = add_residual_lag_features(master)
    return master, yearly_mean_rev, yearly_mean_cogs


def make_lgbm_model() -> lgb.LGBMRegressor:
    return lgb.LGBMRegressor(
        n_estimators=2000,
        learning_rate=0.02,
        num_leaves=31,
        min_child_samples=15,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        verbose=-1,
    )


def train_residual_model(
    master: pd.DataFrame,
    target_col: str,
    feature_cols: list[str],
    train_start: str,
    train_end: str,
    val_start: str | None = None,
    val_end: str | None = None,
):
    df = master.dropna(subset=feature_cols + [target_col]).copy()
    train_mask = (df['Date'] >= train_start) & (df['Date'] <= train_end)
    train_df = df.loc[train_mask].copy()
    model = make_lgbm_model()

    if val_start is None or val_end is None:
        model.fit(train_df[feature_cols], train_df[target_col])
        return model, train_df, None

    val_mask = (df['Date'] >= val_start) & (df['Date'] <= val_end)
    val_df = df.loc[val_mask].copy()
    model.fit(
        train_df[feature_cols],
        train_df[target_col],
        eval_set=[(val_df[feature_cols], val_df[target_col])],
        callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(500)],
    )
    return model, train_df, val_df


def grid_search_revenue_blend(
    actual_2022: pd.DataFrame,
    seasonal: pd.DataFrame,
    pred_resid_2022: pd.DataFrame,
    yearly_mean_rev: pd.Series,
    yoy_candidates: dict[str, float],
    weight_grid: list[float],
):
    base = actual_2022[['Date', 'Revenue', 'month', 'day']].merge(
        seasonal[['month', 'day', 'rev_norm']], on=['month', 'day'], how='left'
    )
    base['rev_norm'] = base['rev_norm'].fillna(1.0)
    base = base.merge(pred_resid_2022, on='Date', how='left')

    results = []
    for yoy_label, yoy_value in yoy_candidates.items():
        seasonal_pred = base['rev_norm'] * (yearly_mean_rev[2021] * yoy_value)
        for weight in weight_grid:
            prediction = (seasonal_pred + weight * base['pred_resid_rev']).clip(0)
            mae = mean_absolute_error(base['Revenue'], prediction)
            results.append({
                'yoy_label': yoy_label,
                'yoy_value': yoy_value,
                'w_lgbm': weight,
                'mae': mae,
            })

    results_df = pd.DataFrame(results).sort_values('mae').reset_index(drop=True)
    best_config = results_df.iloc[0].to_dict()
    return results_df, best_config


def build_future_frame(start_date: str, end_date: str, promo_calendar: pd.DataFrame) -> pd.DataFrame:
    future = pd.DataFrame({'Date': pd.date_range(start_date, end_date, freq='D')})
    future['year'] = future['Date'].dt.year
    future['month'] = future['Date'].dt.month
    future['day'] = future['Date'].dt.day
    future = add_calendar_features(future)
    future = add_promo_features(future, promo_calendar)
    return future


def attach_future_lag_features(future: pd.DataFrame, master: pd.DataFrame) -> pd.DataFrame:
    history = master[['Date', 'residual_rev', 'residual_cogs']].copy()
    scaffold = pd.DataFrame({
        'Date': future['Date'],
        'residual_rev': np.nan,
        'residual_cogs': np.nan,
    })
    extended = pd.concat([history, scaffold], ignore_index=True)
    extended = add_residual_lag_features(extended)
    lag_cols = ['Date'] + REV_LAG_FEATURES + COGS_LAG_FEATURES
    future_with_lags = future.merge(extended[lag_cols], on='Date', how='left')
    return future_with_lags


def build_yearly_means_for_forecast(
    yearly_mean_rev: dict[int, float],
    yearly_mean_cogs: dict[int, float],
    yoy_value: float,
):
    rev_2023 = yearly_mean_rev[2022] * yoy_value
    rev_2024 = rev_2023 * yoy_value
    cogs_2023 = yearly_mean_cogs[2022] * yoy_value
    cogs_2024 = cogs_2023 * yoy_value
    return (
        {2023: rev_2023, 2024: rev_2024},
        {2023: cogs_2023, 2024: cogs_2024},
    )


def add_monthly_cogs_cap(future: pd.DataFrame, stable_sales: pd.DataFrame) -> pd.DataFrame:
    monthly_ratio = (
        stable_sales.assign(cogs_ratio=stable_sales['COGS'] / stable_sales['Revenue'])
        .groupby('month', as_index=False)['cogs_ratio']
        .median()
    )
    capped = future.merge(monthly_ratio, on='month', how='left')
    capped['pred_COGS'] = np.minimum(
        capped['pred_COGS'],
        capped['pred_Revenue'] * capped['cogs_ratio'] * 1.05,
    ).clip(0)
    return capped


In [3]:
sales = load_sales(DATA_DIR)
stable_sales, seasonal, monthly_profile = build_seasonal_profile(sales, STABLE_YEARS)
promo_calendar = build_promo_calendar()
yoy_candidates = get_yoy_candidates(sales.groupby('year')['Revenue'].mean())

print(f'Sales: {sales["Date"].min().date()} -> {sales["Date"].max().date()} ({len(sales)} rows)')
print(f'Seasonal profile rows: {len(seasonal)}')
print(f'Revenue norm std on stable period: {stable_sales["rev_norm"].std():.4f}')
print()
print('Monthly seasonal profile (Revenue norm):')
print(monthly_profile[['month', 'rev_norm']].to_string(index=False))
print()
print('YoY candidates:')
for label, value in yoy_candidates.items():
    print(f'  {label:14s}: {value:.3f}')


Sales: 2012-07-04 -> 2022-12-31 (3833 rows)
Seasonal profile rows: 366
Revenue norm std on stable period: 0.5555

Monthly seasonal profile (Revenue norm):
 month  rev_norm
     1  0.553174
     2  0.817418
     3  1.304950
     4  1.553935
     5  1.500513
     6  1.396847
     7  1.047833
     8  1.093433
     9  0.892335
    10  0.753765
    11  0.554269
    12  0.528639

YoY candidates:
  cagr_2yr      : 1.055
  last_1yr      : 1.121
  flat          : 1.000
  sample_implied: 0.971
  avg_3         : 1.059


In [4]:
master, yearly_mean_rev, yearly_mean_cogs = build_master_table(sales, seasonal, promo_calendar)
revenue_features = [col for col in BASE_FEATURES + REV_LAG_FEATURES if col in master.columns]
cogs_features = [col for col in BASE_FEATURES + COGS_LAG_FEATURES if col in master.columns]

print(f'Master shape: {master.shape}')
print(f'Residual Revenue mean/std: {master["residual_rev"].mean():,.0f} / {master["residual_rev"].std():,.0f}')
print(f'Revenue features ({len(revenue_features)}): {revenue_features}')
print(f'COGS features ({len(cogs_features)}): {cogs_features}')


Master shape: (3833, 40)
Residual Revenue mean/std: 37,056 / 1,213,642
Revenue features (24): ['dow', 'month', 'day', 'quarter', 'week', 'is_weekend', 'days_to_eom', 'is_eom', 'is_bom', 'is_vn_holiday', 'is_tet', 'is_pre_tet', 'is_post_tet', 'year_parity', 'is_promo', 'promo_discount', 'is_urban', 'is_yearend', 'is_midyear', 'is_spring', 'resid_rev_lag364', 'resid_rev_lag365', 'resid_rev_roll7', 'resid_rev_roll30']
COGS features (24): ['dow', 'month', 'day', 'quarter', 'week', 'is_weekend', 'days_to_eom', 'is_eom', 'is_bom', 'is_vn_holiday', 'is_tet', 'is_pre_tet', 'is_post_tet', 'year_parity', 'is_promo', 'promo_discount', 'is_urban', 'is_yearend', 'is_midyear', 'is_spring', 'resid_cogs_lag364', 'resid_cogs_lag365', 'resid_cogs_roll7', 'resid_cogs_roll30']


In [5]:
print('Training residual model for Revenue...')
lgbm_rev, train_rev_df, val_rev_df = train_residual_model(
    master,
    target_col='residual_rev',
    feature_cols=revenue_features,
    train_start=TRAIN_START,
    train_end=TRAIN_END,
    val_start=VAL_START,
    val_end=VAL_END,
)
print(f'Train rows: {len(train_rev_df)} | Val rows: {len(val_rev_df)}')
print()
print('Training residual model for COGS...')
lgbm_cogs, train_cogs_df, val_cogs_df = train_residual_model(
    master,
    target_col='residual_cogs',
    feature_cols=cogs_features,
    train_start=TRAIN_START,
    train_end=TRAIN_END,
    val_start=VAL_START,
    val_end=VAL_END,
)
print(f'Train rows: {len(train_cogs_df)} | Val rows: {len(val_cogs_df)}')


Training residual model for Revenue...
Train rows: 2922 | Val rows: 365

Training residual model for COGS...
Train rows: 2922 | Val rows: 365


In [6]:
actual_2022 = sales.loc[sales['year'] == 2022, ['Date', 'Revenue', 'month', 'day']].copy()
pred_resid_2022 = val_rev_df.loc[val_rev_df['Date'].dt.year == 2022, ['Date']].copy()
pred_resid_2022['pred_resid_rev'] = lgbm_rev.predict(val_rev_df.loc[val_rev_df['Date'].dt.year == 2022, revenue_features])

validation_table, best_config = grid_search_revenue_blend(
    actual_2022=actual_2022,
    seasonal=seasonal,
    pred_resid_2022=pred_resid_2022,
    yearly_mean_rev=sales.groupby('year')['Revenue'].mean(),
    yoy_candidates=yoy_candidates,
    weight_grid=LGBM_WEIGHT_GRID,
)

print('Top validation configs:')
print(validation_table.head(10).to_string(index=False))
print()
print(
    f"Best config -> YoY: {best_config['yoy_label']} ({best_config['yoy_value']:.3f}), "
    f"w_lgbm: {best_config['w_lgbm']:.1f}, MAE: {best_config['mae']:,.0f}"
)


Top validation configs:
yoy_label  yoy_value  w_lgbm           mae
 last_1yr   1.121481     1.0 463381.857499
 last_1yr   1.121481     0.8 464418.327292
    avg_3   1.058715     1.0 465698.436105
 last_1yr   1.121481     0.6 465758.984818
 last_1yr   1.121481     0.5 466608.918317
 cagr_2yr   1.054665     1.0 467493.703159
 last_1yr   1.121481     0.4 467545.649760
    avg_3   1.058715     0.8 468399.846359
 last_1yr   1.121481     0.2 469844.591918
 cagr_2yr   1.054665     0.8 470251.250474

Best config -> YoY: last_1yr (1.121), w_lgbm: 1.0, MAE: 463,382


In [7]:
print('Retraining residual models on full 2014-2022 window...')
lgbm_rev_full, full_train_rev_df, _ = train_residual_model(
    master,
    target_col='residual_rev',
    feature_cols=revenue_features,
    train_start=TRAIN_START,
    train_end=FULL_TRAIN_END,
)
lgbm_cogs_full, full_train_cogs_df, _ = train_residual_model(
    master,
    target_col='residual_cogs',
    feature_cols=cogs_features,
    train_start=TRAIN_START,
    train_end=FULL_TRAIN_END,
)
print(f'Revenue retrain rows: {len(full_train_rev_df)}')
print(f'COGS retrain rows: {len(full_train_cogs_df)}')


Retraining residual models on full 2014-2022 window...
Revenue retrain rows: 3287
COGS retrain rows: 3287


In [8]:
BEST_YOY = float(best_config['yoy_value'])
W_LGBM = float(best_config['w_lgbm'])
YEARLY_MEAN_REV, YEARLY_MEAN_COGS = build_yearly_means_for_forecast(
    yearly_mean_rev=yearly_mean_rev,
    yearly_mean_cogs=yearly_mean_cogs,
    yoy_value=BEST_YOY,
)

future = build_future_frame(FC_START, FC_END, promo_calendar)
future = attach_future_lag_features(future, master)
future = future.merge(seasonal, on=['month', 'day'], how='left')
future['rev_norm'] = future['rev_norm'].fillna(1.0)
future['cogs_norm'] = future['cogs_norm'].fillna(1.0)

for col in revenue_features + cogs_features:
    if col in future.columns:
        future[col] = future[col].fillna(0)

future['ym_rev'] = future['year'].map(YEARLY_MEAN_REV)
future['ym_cogs'] = future['year'].map(YEARLY_MEAN_COGS)
future['seasonal_rev'] = (future['rev_norm'] * future['ym_rev']).clip(0)
future['seasonal_cogs'] = (future['cogs_norm'] * future['ym_cogs']).clip(0)
future['resid_pred_rev'] = lgbm_rev_full.predict(future[revenue_features])
future['resid_pred_cogs'] = lgbm_cogs_full.predict(future[cogs_features])
future['pred_Revenue'] = (future['seasonal_rev'] + W_LGBM * future['resid_pred_rev']).clip(0)
future['pred_COGS'] = (future['seasonal_cogs'] + W_LGBM * future['resid_pred_cogs']).clip(0)
future = add_monthly_cogs_cap(future, stable_sales)

print(f'Forecast rows: {len(future)}')
print(f'Yearly mean Revenue -> 2023: {YEARLY_MEAN_REV[2023]:,.0f}, 2024: {YEARLY_MEAN_REV[2024]:,.0f}')
print(f'Negative gross-profit days: {(future["pred_Revenue"] < future["pred_COGS"]).sum()}')


Forecast rows: 548
Yearly mean Revenue -> 2023: 3,594,111, 2024: 4,030,725
Negative gross-profit days: 11


In [9]:
submission = future[['Date', 'pred_Revenue', 'pred_COGS']].copy()
submission.columns = ['Date', 'Revenue', 'COGS']
submission['Date'] = submission['Date'].dt.strftime('%Y-%m-%d')
submission.to_csv(OUT_FILE, index=False)

print(f'Saved: {OUT_FILE} ({len(submission)} rows)')
print(submission.head(10).to_string(index=False))


Saved: submission_main_pipeline_refactored.csv (548 rows)
      Date      Revenue         COGS
2023-01-01 3.096094e+06 2.633558e+06
2023-01-02 1.109072e+06 9.433839e+05
2023-01-03 9.610960e+05 6.662597e+05
2023-01-04 1.358016e+06 1.155138e+06
2023-01-05 1.390838e+06 1.108744e+06
2023-01-06 1.790476e+06 1.501918e+06
2023-01-07 1.640159e+06 1.395129e+06
2023-01-08 2.214493e+06 1.798790e+06
2023-01-09 2.166027e+06 1.630742e+06
2023-01-10 2.187544e+06 1.860739e+06


## Refactor summary
- Đã bỏ `orders`, `web`, import lặp, block debug/export variants chỉ phục vụ thử nghiệm.
- Toàn bộ calendar / promo / lag engineering giờ đi qua helper chung nên train và forecast không còn lệch logic.
- Việc validate, retrain và forecast được tách thành các cell độc lập, dễ đọc và dễ thay từng phần hơn.
